Decision Tree Model

In [2]:
import numpy as np
import pandas as pd

In [64]:
# Feature selection
df = pd.read_csv("../data/processed/gallstone_all_preprocessed.csv")

key_features = ['Vitamin D', 'Lean Mass (LM) (%)', 'Hemoglobin (HGB)', 'Bone Mass (BM)', 'Extracellular Water (ECW)', 'C-Reactive Protein (CRP)', 'Total Body Fat Ratio (TBFR) (%)', 'Total Fat Content (TFC)', 'Hyperlipidemia', 'High Density Lipoprotein (HDL)']
selected_features = df[key_features]

In [65]:
# Load training and test data 

# Target column
target_col = "Gallstone Status" 

# File paths for datasets
train_file = "../data/processed/gallstone_train_preprocessed.csv"
test_file = "../data/processed/gallstone_test_preprocessed.csv"



# Load the data into two separate DataFrames
train_df = pd.read_csv(train_file).drop(columns=['Set'])
test_df = pd.read_csv(test_file).drop(columns=['Set'])


print(f"Training data loaded with {train_df.shape[1]} columns.")

#train_df.head()


Training data loaded with 39 columns.


In [66]:
# Training set separation
# X_train: All feature columns (inputs)
X_train = train_df.drop(columns=[target_col])

# y_train: The target column (output)
y_train = train_df[target_col]

# Testing set separation
# X_test: All feature columns for validation
X_test = test_df.drop(columns=[target_col])

# y_test: The target column for validation
y_test = test_df[target_col]

print("\nData separated successfully:")
print(f"X_train shape (Features): {X_train.shape}")
print(f"y_train shape (Labels):   {y_train.shape}")


Data separated successfully:
X_train shape (Features): (255, 38)
y_train shape (Labels):   (255,)


In [67]:
# Import Decision Tree classification model and evaluation metrics to assess the model

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix, log_loss
import time

# Initialize the model 

random_seed = 42

decisionTree_model = DecisionTreeClassifier(
    # Limit how deep the tree can grow to prevent overfitting
    max_depth = 5,           
    # Minimum number of samples required to be at a leaf node
    min_samples_leaf = 20,   
    # Use Gini impurity as a quality measure of a split in the tree
    criterion = 'gini',      
    random_state = random_seed 
)

print("Decision Tree Classifier initialized with regularization parameters")

Decision Tree Classifier initialized with regularization parameters


In [68]:
# Train the model

start_time = time.time()

# Fit the model to the training data
decisionTree_model.fit(X_train, y_train)

end_time = time.time()
print(f"Model trained successfully in {end_time - start_time:.4f} seconds.")

Model trained successfully in 0.0035 seconds.


In [79]:
# Use the model to predict Gallstone Status and Evaluate 

# Get Gallstone Status predictions (0 or 1)
y_pred = decisionTree_model.predict(X_test)

# Get prediction probabilities for Gallstone Status = 1, used for AUC score (metric used to measure binary classification perfomance)
y_prob = decisionTree_model.predict_proba(X_test)[:, 1]

# Decision Tree Model - Results Report
# Base Metrics
accuracy = accuracy_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_prob)

# Additional Metrics
# Log Loss requires the prediction probabilities for ALL classes, which predict_proba returns
y_proba_all = decisionTree_model.predict_proba(X_test)
logloss = log_loss(y_test, y_proba_all)

# Confusion Matrix and Specificity (requires the raw matrix)
cm = confusion_matrix(y_test, y_pred)

# The matrix elements:
# cm[0, 0] = True Negatives (TN)
# cm[0, 1] = False Positives (FP)
# cm[1, 0] = False Negatives (FN)
# cm[1, 1] = True Positives (TP)

TN = cm[0, 0]
FP = cm[0, 1]

# Specificity (True Negative Rate) = TN / (TN + FP)
# Measures how well the model identifies true negative cases (Status 0)
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

# Print report

print("\nDecision Tree Model Performance")

# 1. Classification Report (Precision, Recall, F1-Score)
print("Classification Report:\n", classification_report(y_test, y_pred))

# 2. Key Summary Metrics
print(f"Accuracy: {100*accuracy:.2f}%")
print(f"AUC Score: {100*auc_score:.2f}%")
print(f"Log Loss (Calibration): {logloss:.4f}")
print(f"Specificity (True Negative Rate): {100*specificity:.2f}%")

# 3. Confusion Matrix (Raw Counts)
print("\nConfusion Matrix (Raw Counts):")
print(f"   Predicted 0  |  Predicted 1")
print(f"Actual 0:  {cm[0, 0]:<4} |  {cm[0, 1]}")
print(f"Actual 1:  {cm[1, 0]:<4} |  {cm[1, 1]}")


Decision Tree Model Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.69      0.69        32
           1       0.69      0.69      0.69        32

    accuracy                           0.69        64
   macro avg       0.69      0.69      0.69        64
weighted avg       0.69      0.69      0.69        64

Accuracy: 68.75%
AUC Score: 75.63%
Log Loss (Calibration): 0.5453
Specificity (True Negative Rate): 68.75%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  22   |  10
Actual 1:  10   |  22


##### Decision Tree Model Evaluation: Baseline Results

This analysis summarizes the performance of the initial, untuned Decision Tree Classifier on the test dataset (N=64 samples, perfectly balanced).

---

##### 1. Explanation of Evaluation Metrics

| Metric | Interpretation | Desired Value |
| :--- | :--- | :--- |
| **Accuracy** | Overall proportion of correct predictions (True Positives + True Negatives) out of all test cases. | Closer to 100% |
| **Precision** | Of all cases predicted as **Positive (Gallstones)**, how many were actually Gallstones? (Focuses on minimizing False Positives). | Closer to 1.0 |
| **Recall** | Of all cases that were **truly Positive (Gallstones)**, how many were correctly identified? (Focuses on minimizing False Negatives). | Closer to 1.0 |
| **F1-Score** | The harmonic mean of Precision and Recall. Provides a single measure that balances both metrics. | Closer to 1.0 |
| **AUC Score** | **Area Under the ROC Curve**. Measures the model's ability to distinguish between the two classes across all possible thresholds (not just the single 0.5 threshold used to get a final prediction). | Closer to 100% |
| **Specificity** | The True Negative Rate. Measures the proportion of cases correctly identified as **Negative (No Gallstones)**. | Closer to 100% |
| **Log Loss** | A measure of the error based on the model's predicted probabilities. Penalizes confident, incorrect predictions heavily. | Closer to 0.0 |

---

##### 2. Discussion of Baseline Results
 > **Context:** The Decision Tree first outputs a probability of Gallstone Status (0.0 to 1.0) and then converts this to a final **hard class prediction** (0 or 1) using the default threshold of 0.5.

| Metric | Result | Interpretation |
| :--- | :--- | :--- |
| **Accuracy** | **68.75%** | Solid baseline; significantly better than 50% chance. |
| **AUC Score** | **75.63%** | Strong discriminator; the probabilities are more reliable than the hard class predictions. |
| **F1-Score (0 & 1)** | **0.69** | Performance is perfectly balanced across both classes. |
| **Log Loss** | **0.5453** | Low error rate, indicating the model's probability predictions are moderately well-calibrated. |
| **Specificity** | **68.75%** | The model correctly ruled out Gallstones 68.75% of the time. |
| **Confusion Matrix** | **TP: 22, TN: 22** | $22$ samples were correctly classified in each category, while $10$ samples in each category were misclassified (FP: 10, FN: 10). |

The uniform results across Precision, Recall, and Specificity ($\approx 69\%$) are characteristic of a model that is currently **under-fitting** or overly simple. The Decision Tree, by default, was initialized with heavy regularization (like a low `max_depth`) to prioritize robustness over maximum performance. The good AUC score ($\mathbf{75.63\%}$) suggests there is significant potential to increase the hard-classification metrics (Accuracy, F1-Score) through tuning.

---

##### 3. Next Steps: Model Fine-Tuning

The current focus shifts from validation to **optimization** through systematic hyperparameter search to maximize the model's generalization ability.

1.  **Objective:** The primary goal is to find the optimal balance between the model's complexity and its performance, typically by maximizing the **AUC Score**.

2.  **Hyperparameter Search:** Implement **`sklearn.model_selection.GridSearchCV`** to explore a systematic range of regularization parameters.

3.  **Key Parameters to Tune:**
    * **`max_depth`**: To allow the tree to learn more complex relationships.
    * **`min_samples_leaf`**: To control the size and noise tolerance of the resulting leaf nodes.
    * **`criterion`**: Compare the performance when using `'gini'` versus `'entropy'`.

4.  **Model Comparison:** Use the optimized Decision Tree to set a high benchmark for comparison against the other planned models, such as Logistic Regression and Random Forest.